# 4. Model Development

## Pre-processing

In [1]:
# Libraries imported for this notebook.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [2]:
# Read EDA_Data.xlsx into a dataframe, formatted dates, and indexed dates.

df = pd.read_excel('/Users/NJahns/Desktop/Bootcamp/Capstone_Two/Edited_Data/Pre_Process_Train.xlsx', parse_dates=True, index_col=[0])

In [3]:
# Looked at shape

df.shape

(2272, 555)

In [4]:
# Defined Xs and y.

X_all = df.drop(columns=['Sludge Volume Index']) # Original, differenced, and lagged versions of features
X_no_SVI = df.drop(columns=df.columns[df.columns.str.contains('Sludge Volume Index')]) # Original, differenced, and lagged versions of features except SVI version
X_no_lag = df.drop(columns=df.filter(like='lag').columns.tolist() + ['Sludge Volume Index']) # Original, differenced, and no lagged versions of features
X_no_orig = df[df.columns[df.columns.str.contains('_')]]# Differenced and lagged versions of features
X_no_orig_or_SVI = df[[col for col in df.columns if '_' in col and 'Sludge Volume Index' not in col]].copy() # Differenced and lagged versions of features except SVI
X_orig = df[[col for col in df.columns if '_' not in col and col != 'Sludge Volume Index']].copy() # Original features
y = df['Sludge Volume Index'] # Target variable for all models

I created multiple versions of X, the independent variables, with different combinations of including or not including original data, lagged data, and lagged SVI data to try in each model.

In [5]:
# Defined number of splits for time series cross-validation.

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

I used TimeSeriesSplit as a cross-validator because it is specifically designed for time series data. The dataset is split into consecutive folds, where each fold is a superset of the previous one, ensuring that the model is trained on data that retains the temporal nature of time series data. In addition, these methods can also check for autocorrelation.

Models capable of making predictions from time series data were essential for this project. I chose to explore linear regression, random forest, and ARIMA models.

## Linear Regression

In [6]:
# Created funtion to perform cross-validation on folds and calculate MSE and Adj R2

# Defined function
def lrcv(X, y, tscv): # linear regression cross validation
    # Initialize empty lists to store scores and predictions
    lr_mse_scores = []
    lr_predictions = []
    lr_adj_r2_scores = []
    # Iterate over each fold for cross-validation using the time series cross-validation splitter (tscv)    
    for train_index, test_index in tscv.split(X):
        # Split data into training and test sets        
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # Instantiate a linear regression model
        lr_model = LinearRegression()
        # Fit the linear regression model to the training data
        lr_model.fit(X_train, y_train)
        # Make predictions on the test set
        lr_pred = lr_model.predict(X_test)
        # Store predictions
        lr_predictions.extend(lr_pred)
        # Calculate Mean Squared Error (MSE) and append to list
        lr_mse_scores.append(mean_squared_error(y_test, lr_pred))
        # Calculate R^2 score
        r2 = r2_score(y_test, lr_pred)
        n = len(X_test)  # Number of observations
        k = X_train.shape[1]  # Number of predictors
        adj_r2 = 1 - (1 - r2) * ((n - 1) / (n - k - 1))
        # Append adjusted R^2 score to list
        lr_adj_r2_scores.append(adj_r2)
    # Calculate average MSE and average adjusted R^2 score across all folds
    lr_avg_mse = np.mean(lr_mse_scores)
    lr_avg_adj_r2 = np.mean(lr_adj_r2_scores)
    # Return average MSE and average adjusted R^2 score, and predictions
    return lr_avg_mse, lr_avg_adj_r2, lr_predictions

In [7]:
# Ran function with original, differenced, and lagged versions of features
lr_X_all_avg_mse, lr_X_all_avg_adj_r2, lr_X_all_predictions = lrcv(X_all, y, tscv)

In [8]:
# Ran function with original, differenced, and lagged versions of features except SVI version
lr_X_no_SVI_avg_mse, lr_X_no_SVI_avg_adj_r2, lr_X_no_SVI_predictions = lrcv(X_no_SVI, y, tscv)

In [9]:
# Ran function with original, differenced, and no lagged versions of features
lr_X_no_lag_avg_mse, lr_X_no_lag_avg_adj_r2, lr_X_no_lag_predictions = lrcv(X_no_lag, y, tscv)

In [10]:
# Ran function with differenced and lagged versions of features
lr_X_no_orig_avg_mse, lr_X_no_orig_avg_adj_r2, lr_X_no_orig_predictions = lrcv(X_no_orig, y, tscv)

In [11]:
# Ran function with differenced and lagged versions of features except SVI
lr_X_no_orig_or_SVI_avg_mse, lr_X_no_orig_or_SVI_avg_adj_r2, lr_X_no_orig_or_SVI_predictions = lrcv(X_no_orig_or_SVI, y, tscv)

In [12]:
# Ran function with original features
lr_X_orig_avg_mse, lr_X_orig_avg_adj_r2, lr_X_orig_predictions = lrcv(X_orig, y, tscv)

MSE was calculated to measures the error between the model's predictions and the actual values. R squared was calculated to measures the proportion of the variance in SVI that is explained by the features. This value was then adjusted to account for the large number of predictors in the model. This was done because the R-squared value of a model can increase as predictors are added to a model even if they are not contributing to the explanatory power of the model. Adjusting the R-squared corrects this.

## Random Forest

In [13]:
# Created funtion to perform cross-validation on folds and calculate MSE and Adj R2

# Defined function
def rfcv(X, y, tscv): # Random forest cross validation
    # Initialize empty lists to store scores and predictions
    rf_mse_scores = []
    rf_predictions = []
    rf_adj_r2_scores = []
    # Iterate over each fold for cross-validation using the time series cross-validation splitter (tscv)    
    for train_index, test_index in tscv.split(X):
        # Split data into training and test sets        
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # Instantiate a random forest model
        rf_model = RandomForestRegressor()
        # Fit the random forest model to the training data
        rf_model.fit(X_train, y_train)
        # Make predictions on the test set
        rf_pred = rf_model.predict(X_test)
        # Store predictions
        rf_predictions.extend(rf_pred)
        # Calculate Mean Squared Error (MSE) and append to list
        rf_mse_scores.append(mean_squared_error(y_test, rf_pred))
        # Calculate R^2 score
        r2 = r2_score(y_test, rf_pred)
        n = len(X_test)  # Number of observations
        k = X_train.shape[1]  # Number of predictors
        adj_r2 = 1 - (1 - r2) * ((n - 1) / (n - k - 1))
        # Append adjusted R^2 score to list
        rf_adj_r2_scores.append(adj_r2)
    # Calculate average MSE and average adjusted R^2 score across all folds
    rf_avg_mse = np.mean(rf_mse_scores)
    rf_avg_adj_r2 = np.mean(rf_adj_r2_scores)
    # Return average MSE and average adjusted R^2 score, and predictions
    return rf_avg_mse, rf_avg_adj_r2, rf_predictions

In [14]:
# Ran function with original, differenced, and lagged versions of features
rf_X_all_avg_mse, rf_X_all_avg_adj_r2, rf_X_all_predictions = rfcv(X_all, y, tscv)

In [15]:
# Ran function with original, differenced, and lagged versions of features except SVI version
rf_X_no_SVI_avg_mse, rf_X_no_SVI_avg_adj_r2, rf_X_no_SVI_predictions = rfcv(X_no_SVI, y, tscv)

In [16]:
# Ran function with original, differenced, and no lagged versions of features
rf_X_no_lag_avg_mse, rf_X_no_lag_avg_adj_r2, rf_X_no_lag_predictions = rfcv(X_no_lag, y, tscv)

In [17]:
# Ran function with differenced and lagged versions of features
rf_X_no_orig_avg_mse, rf_X_no_orig_avg_adj_r2, rf_X_no_orig_predictions = rfcv(X_no_orig, y, tscv)

In [18]:
# Ran function with differenced and lagged versions of features except SVI
rf_X_no_orig_or_SVI_avg_mse, rf_X_no_orig_or_SVI_avg_adj_r2, rf_X_no_orig_or_SVI_predictions = rfcv(X_no_orig_or_SVI, y, tscv)

In [19]:
# Ran function with original features
rf_X_orig_avg_mse, rf_X_orig_avg_adj_r2, rf_X_orig_predictions = rfcv(X_orig, y, tscv)

## ARIMAX

In [55]:
# Created funtion to perform cross-validation on folds and calculate MSE and Adj R2
from pmdarima.arima import auto_arima

# Modified function with auto ARIMA
def arimax_cv(y, exog, tscv): # ARIMAX cross validation
    # Initialize empty lists to store scores and predictions
    arimax_mse_scores = []
    arimax_predictions = []
    arimax_adj_r2_scores = []    
    # Iterate over each fold for cross-validation using the time series cross-validation splitter (tscv)    
    for train_index, test_index in tscv.split(y):
        # Split data into training and test sets
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        exog_train, exog_test = exog.iloc[train_index], exog.iloc[test_index]
        # Use auto ARIMA to find the best ARIMA parameters
        arimax_model = auto_arima(y_train, exogenous=exog_train)
        # Fit the ARIMAX model to the training data
        arimax_model_fit = arimax_model.fit(y_train, exogenous=exog_train)
        # Make predictions on the test set
        arimax_pred = arimax_model_fit.predict(n_periods=len(y_test), exogenous=exog_test)
        # Store predictions
        arimax_predictions.append(arimax_pred)
        # Calculate Mean Squared Error (MSE) and append to list
        arimax_mse_scores.append(mean_squared_error(y_test, arimax_pred))        
        # Calculate R^2 score
        r2 = r2_score(y_test, arimax_pred)
        n = len(y_test)  # Number of observations
        k = 1 + exog_train.shape[1]  # Number of predictors (1 for the lagged values of the target variable and additional ones for exogenous variables)
        adj_r2 = 1 - (1 - r2) * ((n - 1) / (n - k - 1))        
        # Append adjusted R^2 score to list
        arimax_adj_r2_scores.append(adj_r2)    
    # Calculate average MSE and average adjusted R^2 score across all folds
    arimax_avg_mse = np.mean(arimax_mse_scores)
    arimax_avg_adj_r2 = np.mean(arimax_adj_r2_scores)    
    # Return average MSE and average adjusted R^2 score, and predictions
    return arimax_avg_mse, arimax_avg_adj_r2, arimax_predictions

The ARIMAX model creates differenced and lagged versions of the target variable so the dataset fed to it should not include the SVI differenced and lagged data I created.

In [56]:
# Ran function with original, differenced, and lagged versions of features except SVI version
arimax_X_no_SVI_avg_mse, arimax_X_no_SVI_avg_adj_r2, arimax_X_no_SVI_predictions = arimax_cv(y, X_no_SVI, tscv)

In [57]:
# Ran function with differenced and lagged versions of features except SVI
arimax_X_no_orig_or_SVI_avg_mse, arimax_X_no_orig_or_SVI_avg_adj_r2, arimax_X_no_orig_or_SVI_predictions = arimax_cv(y, X_no_orig_or_SVI, tscv)

In [58]:
# Ran function with original features
arimax_X_orig_avg_mse, arimax_X_orig_avg_adj_r2, arimax_X_orig_predictions = arimax_cv(y, X_orig, tscv)

## Model performance

In [27]:
# Compared model MSE and adj R2 scores.

print(f"Linear Regression using all features (original, differenced, and lagged versions of features)")
print(f"   Average MSE: {lr_X_all_avg_mse}, Average Adj R^2: {lr_X_all_avg_adj_r2}")
print()
print(f"Linear Regression using all features except lagged SVI version")
print(f"   Average MSE: {lr_X_no_SVI_avg_mse}, Average Adj R^2: {lr_X_no_SVI_avg_adj_r2}")
print()
print(f"Linear Regression using original and differenced features (no lagged)")
print(f"   Average MSE: {lr_X_no_lag_avg_mse}, Average Adj R^2: {lr_X_no_lag_avg_adj_r2}")
print()
print(f"Linear Regression using differenced and lagged versions of features (no original)")
print(f"   Average MSE: {lr_X_no_orig_avg_mse}, Average Adj R^2: {lr_X_no_orig_avg_adj_r2}")
print()
print(f"Linear Regression using differenced and lagged versions of features except SVI")
print(f"   Average MSE: {lr_X_no_orig_or_SVI_avg_mse}, Average Adj R^2: {lr_X_no_orig_or_SVI_avg_adj_r2}")
print()
print(f"Linear Regression using original versions of features (no differenced and lagged)")
print(f"   Average MSE: {lr_X_orig_avg_mse}, Average Adj R^2: {lr_X_orig_avg_adj_r2}")
print()

Linear Regression using all features (original, differenced, and lagged versions of features)
   Average MSE: 2.2537144190031678e-20, Average Adj R^2: 1.0

Linear Regression using all features except lagged SVI version
   Average MSE: 1935.983369806506, Average Adj R^2: 38.87135176797169

Linear Regression using original and differenced features (no lagged)
   Average MSE: 1366.2863214973013, Average Adj R^2: -12.712794263467886

Linear Regression using differenced and lagged versions of features (no original)
   Average MSE: 4.850451753667862e-20, Average Adj R^2: 1.0

Linear Regression using differenced and lagged versions of features except SVI
   Average MSE: 1770.3898910606647, Average Adj R^2: 48.87381797058141

Linear Regression using original versions of features (no differenced and lagged)
   Average MSE: 1247.4653789801448, Average Adj R^2: -10.053469692335657



DON'T READ, OLD The average MSE for the linear regression model indicates that the model's predictions are very close to the actual values. Essentially, the linear regression model is performing very well on the dataset, with almost no error. The coefficient of determination indicates  that the Linear Regression model perfectly fits the data, capturing all the variability in SVI using the provided features. In other words, the model explains 100% of the variance in the target variable, which is an ideal scenario and implies an excellent model fit.

In [28]:
# Compared model MSE and adj R2 scores.

print(f"Random forest using all features (original, differenced, and lagged versions of features)")
print(f"   Average MSE: {rf_X_all_avg_mse}, Average Adj R^2: {rf_X_all_avg_adj_r2}")
print()
print(f"Random forest using all features except lagged SVI")
print(f"   Average MSE: {rf_X_no_SVI_avg_mse}, Average Adj R^2: {rf_X_no_SVI_avg_adj_r2}")
print()
print(f"Random forest using original and differenced features (no lagged)")
print(f"   Average MSE: {rf_X_no_lag_avg_mse}, Average Adj R^2: {rf_X_no_lag_avg_adj_r2}")
print()
print(f"Random forest using differenced and lagged versions of features (no original)")
print(f"   Average MSE: {rf_X_no_orig_avg_mse}, Average Adj R^2: {rf_X_no_orig_avg_adj_r2}")
print()
print(f"Random forest using differenced and lagged versions of features except SVI")
print(f"   Average MSE: {rf_X_no_orig_or_SVI_avg_mse}, Average Adj R^2: {rf_X_no_orig_or_SVI_avg_adj_r2}")
print()
print(f"Random forest using original versions of features (no differenced and lagged)")
print(f"   Average MSE: {rf_X_orig_avg_mse}, Average Adj R^2: {rf_X_orig_avg_adj_r2}")
print()

Random forest using all features (original, differenced, and lagged versions of features)
   Average MSE: 31.814088787827615, Average Adj R^2: 1.423651494903392

Random forest using all features except lagged SVI
   Average MSE: 856.5352463217883, Average Adj R^2: 24.898766115058407

Random forest using original and differenced features (no lagged)
   Average MSE: 776.5002157156842, Average Adj R^2: -11.107898066021018

Random forest using differenced and lagged versions of features (no original)
   Average MSE: 29.688403992196992, Average Adj R^2: 1.4587994676064293

Random forest using differenced and lagged versions of features except SVI
   Average MSE: 1165.1548395262062, Average Adj R^2: 34.52540140894059

Random forest using original versions of features (no differenced and lagged)
   Average MSE: 772.5158973062235, Average Adj R^2: -8.974754861690208



DON'T READ, OLD Considering that the mean SVI value in the data is 101 with a max of 235, the random forest model MSE indicates a low level of error in the predictions.The coefficient of determination indicates that the Random Forest model explains about 80% of the variance in the target variable, while the remaining 20% of the variance is unexplained and could be due to factors not accounted for in the model or random variability. In addition, the R^2 value is relatively high, indicating a good fit of the model to the data and a strong ability to explain the variability in SVI using the provided features.

In [59]:
# Compared model MSE and adj R2 scores.

print(f"Random forest using all features except lagged SVI")
print(f"   Average MSE: {arimax_X_no_SVI_avg_mse}, Average Adj R^2: {arimax_X_no_SVI_avg_adj_r2}")
print()
print(f"Random forest using differenced and lagged versions of features except SVI")
print(f"   Average MSE: {arimax_X_no_orig_or_SVI_avg_mse}, Average Adj R^2: {arimax_X_no_orig_or_SVI_avg_adj_r2}")
print()
print(f"Random forest using original versions of features (no differenced and lagged)")
print(f"   Average MSE: {arimax_X_orig_avg_mse}, Average Adj R^2: {arimax_X_orig_avg_adj_r2}")

Random forest using all features except lagged SVI
   Average MSE: 363.6497124050787, Average Adj R^2: 5.958701251197894

Random forest using differenced and lagged versions of features except SVI
   Average MSE: 363.6497124050787, Average Adj R^2: 6.912297645659028

Random forest using original versions of features (no differenced and lagged)
   Average MSE: 363.6497124050787, Average Adj R^2: -1.189739868762603


DON'T READ, OLD The MSE for the ARIMA model indicates poor performance. In combination with the negative R^2 value, these scores indicate that the ARIMA model does not provide a good fit to the data and performs worse than a simple mean prediction model.

In [ ]:
# Defined function to plot actual vs predicted values.

def plot_actual_vs_predicted(actual, predicted, model_name):
    plt.figure(figsize=(10, 6))
    plt.plot(actual.index, actual.values, label='Actual', color='blue')
    plt.plot(actual.index, predicted, label='Predicted', color='red')
    plt.title(f'Actual vs Predicted (Model: {model_name})')
    plt.xlabel('Date')
    plt.ylabel('Sludge Volume Index')
    plt.legend()
    plt.show()

In [ ]:
# Plotted actual vs predicted values for each model.

plot_actual_vs_predicted(y.iloc[len(y) - len(lr_predictions):], lr_predictions, "Linear Regression")
plot_actual_vs_predicted(y.iloc[len(y) - len(rf_predictions):], rf_predictions, "Random Forest")
plot_actual_vs_predicted(y.iloc[len(y) - len(arima_predictions):], arima_predictions, "ARIMA")

Actual and predicted values appear to be identical for the linear regressioin model, moderateley similar for the random forest model, and not at all similar for the ARIMA model.

Overall, the linear regression model is the best predictive model for this project.

## Feature importances

In [ ]:
# Identified the top 10 features with the highest importance.

# Got the feature names
feature_names = X_train.columns

# Created a dictionary mapping feature names to coefficients
feature_coef = dict(zip(feature_names, lr_model.coef_))

# Sorted the dictionary by coefficient values in descending order
sorted_feature_coef = dict(sorted(feature_coef.items(), key=lambda item: abs(item[1]), reverse=True))

# Print the top 10 features along with their coefficients
print("Top 10 Features and Their Coefficients:")
for i, (feature, coef) in enumerate(sorted_feature_coef.items()):
    if i >= 10:
        break
    print(f"{feature}: {coef}")

The coefficients indicate that first order differenced Sludge Volume Index values is the most important feature for predicting the current SVI, followed by SVI from the previous day. ALL top 10 most important features are related to SVI indicating that the other features predict very little.

# Data save

In [ ]:
# Saved to Excel.
df.to_excel('/Users/NJahns/Desktop/Bootcamp/Capstone_Two/Edited_Data/Model_Data.xlsx', index=True)